# Credit Default Risk — Step 4a: The Deployment Model

The Step 2 model uses ~130 columns, some of which Step 3 showed should not drive a credit decision. For deployment we train a **slimmer, fairer model**:

- **Removed:** marital status (protected in many lending rules), application weekday and hour (process artefacts, not borrower behaviour), and age (protected in many jurisdictions, so excluded to be cautious). Gender was already removed in Step 2.
- **Kept:** only inputs an applicant could realistically enter in a form.

Then we measure what this costs in AUC compared with the full model, and save the model plus a **form spec** for the web app.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from features import prepare, split_xy, to_model_frame, EMPLOYED_PLACEHOLDER

DATA_PATH = Path("data/application_train.csv")
RESULTS = Path("results"); MODELS = Path("models")
RANDOM_STATE = 42

## 1. Choose the inputs

In [ ]:
RAW_INPUTS = [
    # Loan and affordability
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "NAME_CONTRACT_TYPE",
    # Credit bureau scores (normalised 0-1)
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    # Employment and background
    "DAYS_EMPLOYED", "NAME_INCOME_TYPE", "ORGANIZATION_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE",
    # Assets and residence
    "FLAG_OWN_CAR", "OWN_CAR_AGE", "FLAG_OWN_REALTY", "REG_CITY_NOT_LIVE_CITY",
]
EXCLUDED = ["CODE_GENDER", "NAME_FAMILY_STATUS", "WEEKDAY_APPR_PROCESS_START", "HOUR_APPR_PROCESS_START", "DAYS_BIRTH"]

raw = pd.read_csv(DATA_PATH)
missing = [c for c in RAW_INPUTS if c not in raw.columns]
assert not missing, f"Columns missing from the data: {missing}"
assert not set(RAW_INPUTS) & set(EXCLUDED)

df = prepare(raw[["SK_ID_CURR", "TARGET"] + RAW_INPUTS])
X, y = split_xy(df)
print(f"Raw inputs: {len(RAW_INPUTS)}   Model features after engineering: {X.shape[1]}")
print(sorted(X.columns))

## 2. Same splits as Step 2

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, stratify=y_train_full, random_state=RANDOM_STATE)
print(f"Train: {len(X_train):,}   Validation: {len(X_val):,}   Test: {len(X_test):,}")

## 3. Train the deployment model

In [ ]:
slim = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.03, max_depth=5, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", enable_categorical=True, max_cat_to_onehot=1,
    eval_metric="auc", early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
)
slim.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)

p_val = slim.predict_proba(X_val)[:, 1]
p_test = slim.predict_proba(X_test)[:, 1]
auc_slim = roc_auc_score(y_test, p_test)

full = json.loads((RESULTS / "xgb_metrics.json").read_text())
print(f"\nFull model (Step 2, ~130 columns):      AUC {full['test_auc']:.4f}")
print(f"Deployment model ({len(RAW_INPUTS)} inputs, fairer): AUC {auc_slim:.4f}   Gini {2 * auc_slim - 1:.4f}")
print(f"Cost of slimming down:                  {auc_slim - full['test_auc']:+.4f} AUC")

## 4. Approval threshold (same 5:1 cost assumption as Step 2)

In [ ]:
COST_BAD_APPROVED, COST_GOOD_REJECTED = 5.0, 1.0

def expected_cost(y_true, probs, t):
    approve = probs < t
    return (COST_BAD_APPROVED * np.sum(approve & (y_true == 1)) +
            COST_GOOD_REJECTED * np.sum(~approve & (y_true == 0))) / len(y_true)

thresholds = np.linspace(0.02, 0.60, 59)
best_t = float(thresholds[int(np.argmin([expected_cost(y_val.values, p_val, t) for t in thresholds]))])

approve = p_test < best_t
print(f"Threshold: approve if default probability < {best_t:.2f}")
print(f"Approval rate: {approve.mean():.1%}")
print(f"Default rate among approved: {y_test[approve].mean():.2%}  (vs {y_test.mean():.2%} if everyone approved)")

## 5. What drives the deployment model? (SHAP)

In [ ]:
contribs = slim.get_booster().predict(xgb.DMatrix(X_test, enable_categorical=True), pred_contribs=True)
mean_abs = pd.Series(np.abs(contribs[:, :-1]).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
mean_abs.head(15)[::-1].plot.barh(ax=ax)
ax.set_xlabel("Mean |SHAP value| (log-odds)"); ax.set_title("Deployment model: what drives risk")
fig.savefig(RESULTS / "deploy_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Build the form spec

For each input: its type, a sensible range or dropdown options, and a default (the median applicant). The web app uses this to build the form, and the API uses it to validate requests.

In [ ]:
FIELD_INFO = {
    "AMT_INCOME_TOTAL": ("Annual income", "money"),
    "AMT_CREDIT": ("Loan amount", "money"),
    "AMT_ANNUITY": ("Annual repayment", "money"),
    "AMT_GOODS_PRICE": ("Price of goods being financed", "money"),
    "NAME_CONTRACT_TYPE": ("Loan type", "category"),
    "EXT_SOURCE_1": ("Credit bureau score 1 (0-1)", "score"),
    "EXT_SOURCE_2": ("Credit bureau score 2 (0-1)", "score"),
    "EXT_SOURCE_3": ("Credit bureau score 3 (0-1)", "score"),
    "DAYS_EMPLOYED": ("Years in current job", "years"),
    "NAME_INCOME_TYPE": ("Income type", "category"),
    "ORGANIZATION_TYPE": ("Employer sector", "category"),
    "OCCUPATION_TYPE": ("Occupation", "category"),
    "NAME_EDUCATION_TYPE": ("Education level", "category"),
    "FLAG_OWN_CAR": ("Owns a car", "category"),
    "OWN_CAR_AGE": ("Car age (years)", "number"),
    "FLAG_OWN_REALTY": ("Owns property", "category"),
    "REG_CITY_NOT_LIVE_CITY": ("Lives in a different city from registered address", "flag"),
}

fields = []
for col in RAW_INPUTS:
    name, kind = FIELD_INFO[col]
    s = raw[col]
    f = {"name": col, "label": name, "type": kind}
    if kind == "category":
        f["options"] = sorted(str(v) for v in s.dropna().unique())
        f["default"] = str(s.mode().iloc[0])
    elif kind == "years":  # stored as negative days; the form asks for years
        yrs = (-s[s != EMPLOYED_PLACEHOLDER] / 365.25).dropna()
        f.update(min=0.0, max=round(float(yrs.quantile(0.99)), 1), default=round(float(yrs.median()), 1))
    elif kind == "flag":
        f.update(options=[0, 1], default=int(s.mode().iloc[0]))
    else:
        v = s.dropna()
        f.update(min=round(float(v.quantile(0.01)), 2), max=round(float(v.quantile(0.99)), 2),
                 default=round(float(v.median()), 2))
    f["required"] = col not in ("EXT_SOURCE_1", "EXT_SOURCE_3", "OWN_CAR_AGE", "OCCUPATION_TYPE")
    fields.append(f)

schema = {
    "model": "slim_xgboost",
    "raw_inputs": RAW_INPUTS,
    "excluded_on_fairness_grounds": EXCLUDED,
    "features": list(X.columns),
    "categories": {c: [str(v) for v in X[c].cat.categories] for c in X.select_dtypes("category").columns},
    "threshold": round(best_t, 2),
    "test_auc": round(auc_slim, 4),
    "form_fields": fields,
}
pd.DataFrame(fields)[["name", "label", "type", "default"]]

## 7. Round-trip check and save

In [ ]:
# Feed one test applicant through the same path the API will use, and confirm the prediction matches.
row_id = X_test.index[0]
api_frame = to_model_frame(raw.loc[[row_id], RAW_INPUTS], schema)
p_api = slim.predict_proba(api_frame)[:, 1][0]
p_nb = slim.predict_proba(X_test.loc[[row_id]])[:, 1][0]
print(f"Notebook prediction {p_nb:.6f}   API-path prediction {p_api:.6f}   Match: {np.isclose(p_api, p_nb)}")

slim.save_model(MODELS / "slim_model.json")
(MODELS / "slim_schema.json").write_text(json.dumps(schema, indent=2))
metrics = {"model": "slim_xgboost", "inputs": len(RAW_INPUTS), "test_auc": round(auc_slim, 4),
           "test_gini": round(2 * auc_slim - 1, 4), "full_model_auc": full["test_auc"],
           "threshold": round(best_t, 2), "approval_rate": round(float(approve.mean()), 4),
           "default_rate_approved": round(float(y_test[approve].mean()), 4)}
(RESULTS / "slim_metrics.json").write_text(json.dumps(metrics, indent=2))
metrics

## Next step
Send back the **AUC comparison**, the **threshold results** and the **round-trip check**. Then Step 4b wraps `slim_model.json` and `slim_schema.json` in an API.